#### Excelデータを1行ずつ処理する方法

用途は、アンケート結果分析など。
主要なキーワードを抽出する処理を自動化

In [5]:
# エクセルデータを一行ずつ処理

#APIリクエストの準備
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv("../.env")  #load_dotenv("../.env") ?

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"


1. PythonプログラムでExcelファイルを読み込む

In [6]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx',
                    sheet_name='アンケート')
# データフレームを表示して確認。（列は指定しなくてよい？
df.head()

,アンケート
0,「使った瞬間、髪がしっとりサラサラになり驚きました。香りも強すぎずリラックスできる香りで気に...
1,「洗い上がりはスッキリするのに、髪が全然きしまず指通りが良くなりました！乾燥しがちな髪質です...
2,「香りが少し甘めで、好みが分かれそうです。個人的には好きですが、甘い香りが苦手な方には少し強...
3,「ボトルのデザインがシンプルでおしゃれなので、バスルームが映える感じがします。使用感も良くて...
4,「使い始めてから頭皮のかゆみが減りました。敏感肌なので合わないシャンプーが多いのですが、これ...


2. アンケートの主要キーワードをLLMで抽出する関数
3. OpenAI APIの呼び出し

In [ ]:
#アンケートの結果テキストを、LLMを使ってキーワード化する関数。

def extract_keywords(text):
    #データをLLM用にテキスト形式に変換
    #プロンプトの作成
    prompt_text = (
        f"以下のアンケート回答から主要なキーワードをカンマ区切りで"
        f"抽出してください。\nアンケート: {str(text)}"
    )

    # openAIの呼び出し （エラー対策の例外処理つき！
    try:
        #opena APIで応答を生成
        response = client.chat.completions.create(
            model = MODEL_NAME,
            messages = [
                {"role": "user", "content": prompt_text}
            ],
            max_tokens=100,   # 応答の長さを調整
            temperature= 0.3,  # 創造性の調整
        )
        # キーワード抽出結果を取得
        keywords = response.choices[0].message.content.strip()
        return keywords

    except Exception as e:
        print(f"API呼び出しでエラーが発生しました：{e}")
        return "エラー"


In [8]:
# テストしてみる
text = extract_keywords(
    "pandasは、Excelファイル（`.xls`、`.xlsx` 形式）の"
    "読み書きに加え、データの集計や変換、フィルタリングと"
    "いった複雑な操作を手軽に行えるライブラリです。"
    )
print(text)

pandas, Excelファイル, 読み書き, データ, 集計, 変換, フィルタリング, 複雑な操作, ライブラリ


4. LLMからの応答を取得し、Pythonで結果を処理

In [ ]:
# 4. 分析結果を受け取ってデータフレームに変換。※数百行あったから時間かかる
# まずはエクセルの B列「キーワード」を初期化

df['キーワード'] = ""

# for文で各アンケート行に対してキーワード抽出を実行
for index, row in df.iterrows():
    survey_text = row['アンケート'] # A列のアンケ内容取得
    #キーワード抽出関数を呼び出し、結果をB列に保存
    keywords = extract_keywords(survey_text)
    df.at[index,'キーワード'] = keywords

# データフレームを表示していったん確認
df.head()

,アンケート,キーワード
0,「使った瞬間、髪がしっとりサラサラになり驚きました。香りも強すぎずリラックスできる香りで気に...,"髪, しっとり, サラサラ, 香り, リラックス, 泡立ち, 少量, 洗える, コスパ, リ..."
1,「洗い上がりはスッキリするのに、髪が全然きしまず指通りが良くなりました！乾燥しがちな髪質です...,"洗い上がり, スッキリ, 髪, きしまず, 指通り, 乾燥, 髪質, シャンプー, 翌朝, ..."
2,「香りが少し甘めで、好みが分かれそうです。個人的には好きですが、甘い香りが苦手な方には少し強...,"香り, 甘め, 好み, 強く感じる, 洗い上がり, 軽やか, 頭皮, すっきり, 夏場, 快適"
3,「ボトルのデザインがシンプルでおしゃれなので、バスルームが映える感じがします。使用感も良くて...,"ボトルデザイン, シンプル, おしゃれ, バスルーム, 使用感, 乾燥, パサつき, しっと..."
4,「使い始めてから頭皮のかゆみが減りました。敏感肌なので合わないシャンプーが多いのですが、これ...,"頭皮のかゆみ, 敏感肌, シャンプー, 刺激が少ない, 使いやすい, 香り, ナチュラル, ..."


In [ ]:
# pandasのメソッド、apply これだけでもできる。
# データは上書きされるけど、一応、列は初期化をする。

df['キーワード'] = ""

df['キーワード'] = df['アンケート'].apply(extract_keywords)


★メモ：apply() メソッドについて

pandasデータの Series(列) や DataFrame(表全体) に対して、
関数を一括で適用できるメソッド。

例
df['列名'].apply(関数)

例
def combine(row): 
return f"{row['名前']}さんは{row['アンケート']}と答えました"
df['まとめ'] = df.apply(combine, axis=1)


5. 結果をExcelファイルなどに書き戻す

In [ ]:
# 5. 結果をExcelファイルに保存
df.to_excel("アンケート分析結果.xlsx", index=False)

""" シート名を指定する差場合は sheet_name="シート名" とする """


これも実際に使うには、ワークフロー化を。

In [14]:
""" ★準備 """
#APIリクエストの準備
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv("../.env")  #load_dotenv("../.env") ?

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"


""" ★AIとのやりとりの関数をつくる """

def extract_keywords(text):
    #データをLLM用にテキスト形式に変換
    #プロンプトの作成
    prompt_text = (
        f"以下のアンケート回答から主要なキーワードをカンマ区切りで"
        f"抽出してください。\nアンケート: {str(text)}"
    )

    # openAIの呼び出し （エラー対策の例外処理つき！
    try:
        #opena APIで応答を生成
        response = client.chat.completions.create(
            model = MODEL_NAME,
            messages = [
                {"role": "user", "content": prompt_text}
            ],
            max_tokens=100,   # 応答の長さを調整
            temperature= 0.3,  # 創造性の調整
        )
        # キーワード抽出結果を取得
        keywords = response.choices[0].message.content.strip()
        return keywords

    except Exception as e:
        print(f"API呼び出しでエラーが発生しました：{e}")
        return "エラー"


""" ★一連の流れをワークフローとしてつくる """

# ワークフロー化
print("処理を開始します。")

# Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='アンケート')

# A列の各行のアンケートに対してキーワード抽出を実行し、B列に書き込む
df['キーワード'] = df['アンケート'].apply(extract_keywords)

# 結果をExcelファイルに保存
df.to_excel("アンケート分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")

処理を開始します。
Excelファイルに分析結果を保存しました。


注意：同じファイル名のデータがあると全上書きされる